# Task 1 — fetch ENTSO-E day-ahead prices (A44) to landing
Ingests day-ahead prices for Poland for each day and saves **one JSON file per day**
to container landing. Idempotency: consistent file name = no duplicates.
Token read from shared secret scope `default2`.

In [0]:
# Parameters (widgets)
dbutils.widgets.text("n_days", "30", "Number of days")
dbutils.widgets.multiselect("bidding_zone", "PL", ["PL", "DE_LU", "FR", "ES", "CZ", "SK", "LT", "PT"], "Bidding zone")
dbutils.widgets.text("secret_scope", "default2", "Secret scope")
dbutils.widgets.text("secret_key", "gabriela-entsoe-token", "Token with key")


N_DAYS       = int(dbutils.widgets.get("n_days"))
BIDDING_ZONE = dbutils.widgets.get("bidding_zone")
ZONES = [z.strip() for z in BIDDING_ZONE.split(",") if z.strip()]
SCOPE        = dbutils.widgets.get("secret_scope")
KEY          = dbutils.widgets.get("secret_key")
ZONE_CODES = {
    "PL":    "10YPL-AREA-----S",
    "DE_LU": "10Y1001A1001A82H",
    "FR":    "10YFR-RTE------C",
    "ES":    "10YES-REE------0",
    "CZ":    "10YCZ-CEPS-----N",
    "SK":    "10YSK-SEPS-----K",
    "LT":    "10YLT-1001A0008Q",
    "PT":    "10YPT-REN------W"
}

LANDING = "/Volumes/dbr_dev/gabrielajaniszews786_bronze/entsoe_landing"
API_URL = "https://web-api.tp.entsoe.eu/api"

In [0]:
print(BIDDING_ZONE)

In [0]:
# Volume for landing
spark.sql("CREATE VOLUME IF NOT EXISTS dbr_dev.gabrielajaniszews786_bronze.entsoe_landing")

In [0]:
%pip install tqdm

In [0]:
import requests, time
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone
import pandas as pd
from tqdm.notebook import tqdm

token = dbutils.secrets.get(scope=SCOPE, key=KEY)

def parse_prices(xml_text):
    """XML A44 -> tuple list (timestamp_utc_iso, price, currency, unit).
    Returns [] if Acknowledgement (no data - weekends, etc.)"""
    root = ET.fromstring(xml_text)
    tag = root.tag.split("}")[-1]
    if tag != "Publication_MarketDocument":     # for instance Acknowledgement_MarketDocument = no data
        return []
    ns = {"ns": root.tag.split("}")[0].strip("{")}
    rows = []
    for ts in root.findall("ns:TimeSeries", ns):
        currency = ts.findtext("ns:currency_Unit.name", namespaces=ns)
        unit     = ts.findtext("ns:price_Measure_Unit.name", namespaces=ns)
        for period in ts.findall("ns:Period", ns):
            start = period.findtext("ns:timeInterval/ns:start", namespaces=ns)
            res   = period.findtext("ns:resolution", namespaces=ns)
            start_dt = datetime.strptime(start, "%Y-%m-%dT%H:%MZ").replace(tzinfo=timezone.utc)
            step = timedelta(minutes=15 if res == "PT15M" else 60)
            for point in period.findall("ns:Point", ns):
                pos   = int(point.findtext("ns:position", namespaces=ns))
                price = float(point.findtext("ns:price.amount", namespaces=ns))
                ts_iso = (start_dt + step * (pos - 1)).strftime("%Y-%m-%dT%H:%M:%SZ")
                rows.append((ts_iso, price, currency, unit))
    return rows


def fetch_day(day, code):
    """Gets one day of prices (A44); supports 429 status code with retry. Returns (status, xml_text)."""
    params = {
        "securityToken": token,
        "documentType": "A44",
        "in_Domain":  code,
        "out_Domain": code,
        "periodStart": day.strftime("%Y%m%d0000"),
        "periodEnd":  (day + timedelta(days=1)).strftime("%Y%m%d0000"),
    }
    for attempt in range(5):
        r = requests.get(API_URL, params=params, timeout=60)
        if r.status_code == 429:            # limit 400/min exceeded — wait and retry
            time.sleep(5 * (attempt + 1))
            continue
        return r.status_code, r.text
    return 429, ""

In [0]:
today = datetime.now(timezone.utc).date()
written, empty, failed = 0, 0, 0

for zone in ZONES:
    print(f"Processing zone {zone}...")
    code = ZONE_CODES[zone]
    for i in tqdm(range(1, N_DAYS + 1)):
        day = datetime.combine(today - timedelta(days=i), datetime.min.time(), tzinfo=timezone.utc)
        day_str = day.strftime("%Y%m%d")
        path = f"{LANDING}/prices_{zone}_{day_str}.json"
        try:
            status, xml_text = fetch_day(day, code)
            if status != 200:
                print(f"{day_str}: HTTP {status} — omitted")
                failed += 1
                continue
            rows = parse_prices(xml_text)
            if not rows:
                print(f"{day_str}: no data (Acknowledgement)")
                empty += 1
                continue
            out = pd.DataFrame(rows, columns=["timestamp_utc", "price", "currency", "unit"])
            out["bidding_zone"] = zone
            out.to_json(path, orient="records", lines=True, force_ascii=False)
            written += 1
        except Exception as e:
            print(f"{day_str}: {e} error")
            failed += 1
        time.sleep(0.3)                            

print(f"\nProcess completed. Saved data: {written}, days with no data: {empty}, errors: {failed}")